# 07 — Fold balance statistics

Integer division in `FoldPlanner._partition` gives every block the same width except the last, which absorbs the remainder when the azimuth extent is not divisible by `n_folds`. This notebook quantifies that imbalance, both for a clean case and for a deliberately non-divisible case, and visualises the per-block and per-split sample counts.

Modules exercised: `pipelines.cross_validation_pipeline.folds.FoldPlanner`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import numpy             as np
import matplotlib.pyplot as plt

try:
    import torch
    torch.manual_seed(0)
except Exception:
    torch = None

SEED = 0
RNG  = np.random.default_rng(SEED)

plt.rcParams.update({
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.titlesize" : 11,
    "axes.labelsize" : 10,
    "image.cmap"     : "viridis",
})

print("Bootstrap complete. Repository root on sys.path:", Path("../..").resolve())


## Two planners: a divisible extent and a non-divisible extent

The divisible extent should yield perfectly equal blocks; the non-divisible extent concentrates the remainder in the final block.

In [ ]:
from configuration.cross_validation_config import CrossValidationConfig
from pipelines.cross_validation_pipeline.folds import FoldPlanner

def make_planner(azimuth_start, azimuth_end, n_folds):
    config                     = CrossValidationConfig()
    config.folds.n_folds       = n_folds
    config.folds.azimuth_start = azimuth_start
    config.folds.azimuth_end   = azimuth_end
    return FoldPlanner(config, range_start=0, range_end=100)

divisible     = make_planner(0, 800, 8)
non_divisible = make_planner(1000, 16000, 8)

def widths(planner):
    return [end - start for start, end in planner.blocks]

print("divisible widths    :", widths(divisible))
print("non-divisible widths:", widths(non_divisible))

## Summarise the balance with simple statistics

In [ ]:
def balance(planner):
    w = np.array(widths(planner))
    return dict(min=int(w.min()), max=int(w.max()), mean=float(w.mean()), std=float(w.std()), spread=int(w.max() - w.min()))

for label, planner in (("divisible", divisible), ("non-divisible", non_divisible)):
    stats = balance(planner)
    print(f"{label:14s}: min={stats['min']} max={stats['max']} mean={stats['mean']:.1f} std={stats['std']:.2f} spread={stats['spread']}")

## Bar chart of per-block widths

Equal-height bars mean a perfectly balanced partition; a single taller bar at the end is the remainder absorbed by the last block.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=False)

for ax, (label, planner) in zip(axes, (("divisible [0, 800)", divisible), ("non-divisible [1000, 16000)", non_divisible))):
    w = widths(planner)
    ax.bar(range(len(w)), w, color="#4878a8", edgecolor="white")
    ax.axhline(np.mean(w), color="#d96c6c", linestyle="--", label="mean width")
    ax.set_title(label)
    ax.set_xlabel("block index")
    ax.set_ylabel("block width (samples)")
    ax.legend()

plt.tight_layout()
plt.show()

## Per-split sample counts for the non-divisible case

We translate the per-fold regions into train / val / test sample counts (azimuth width, since range is constant) and plot them as stacked bars per fold. Test and validation counts vary only when the last block participates.

In [ ]:
plans = non_divisible.plans()

train_counts = [sum(r.azimuth_size for r in plan.split_regions.regions("train")) for plan in plans]
val_counts   = [plan.split_regions.regions("val")[0].azimuth_size  for plan in plans]
test_counts  = [plan.split_regions.regions("test")[0].azimuth_size for plan in plans]

x = np.arange(len(plans))
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.bar(x, train_counts, color="#7bb37b", label="train")
ax.bar(x, val_counts,  bottom=train_counts, color="#f4c879", label="val")
ax.bar(x, test_counts, bottom=np.array(train_counts) + np.array(val_counts), color="#d96c6c", label="test")

ax.set_xticks(x)
ax.set_xticklabels([f"fold {i}" for i in range(len(plans))])
ax.set_ylabel("azimuth samples")
ax.set_title("Per-fold split sample counts (non-divisible extent)")
ax.legend(loc="upper right")
plt.show()

## Expected visual outcome

The divisible partition shows eight identical bars (spread zero). The non-divisible partition shows seven equal bars and one taller final bar carrying the remainder. The stacked per-fold chart shows constant total height with a slightly larger test or validation segment for the folds that include the final, wider block.